<a href="https://colab.research.google.com/github/thofaa/inflation_YoY_prediction/blob/main/Extraction-Code-Inflation-Prediction-MtM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests

In [ ]:
API_KEY = "..."
url = f"https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/0000/var/123/th/126/key/{API_KEY}"

p = requests.get(url=url)
print(p.headers)

{'Date': 'Mon, 31 Aug 2026 21:10:49 GMT', 'Access-Control-Allow-Origin': '*', 'Keep-Alive': 'timeout=60, max=59', 'Connection': 'Keep-Alive', 'Content-Type': 'application/json', 'Set-Cookie': 'TS01f66aea=0167a1c861a61bd8a68cf198d8ac09bee08d796e4761df96fdb8d845dfb4ce8e33dce2f30e9852d5f3a4a85e1f1d6f7ceea70ede8c; Path=/; Domain=.webapi.bps.go.id; HttpOnly, TS7e23ad4f029=0815dd1fcdab28008992b3164e34b467488f695de02bebb0770f5e19f6c7a18de1c0ebb79edd3a0f7fc09e779f21eaec; Max-Age=30; Path=/', 'Transfer-Encoding': 'chunked'}
{'status': 'OK', 'data-availability': 'available', 'last_update': '2026-08-31 07:54:24', 'subject': [{'val': 562, 'label': 'Perbankan, Asuransi dan Finansial'}], 'var': [{'val': 123, 'label': 'Uang Beredar', 'unit': 'Milyar Rupiah', 'subj': 'Keuangan', 'def': '', 'decimal': 2, 'note': '<p><br /></p><p>Sumber : Bank Indonesia</p><p><br /></p>'}], 'turvar': [{'val': '0', 'label': 'Tidak ada'}], 'labelvervar': 'Jenis Uang', 'vervar': [{'val': 1, 'label': 'i. Uang Kartal'}, {'va

# Extract economy report from BPS

In [2]:
import json

initial_bound = 117
end_bound = 127 #plus one for boundary

for y in range(initial_bound, end_bound):
  API_KEY = "..."
  year = y
  url = f"https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/0000/var/123/th/{year}/key/{API_KEY}"

  p = requests.get(url=url)
  print(f"result for year {year} is", p)

  if p.status_code == int(200):
    year_actual = "20" + str(year)[1:3]
    with open(f"SEK_Data_{year_actual}.json", mode="w") as file:
      file.write(json.dumps(p.json(), indent=4))

  print("write is success!\n")

result for year 117 is <Response [200]>
write is success!

result for year 118 is <Response [200]>
write is success!

result for year 119 is <Response [200]>
write is success!

result for year 120 is <Response [200]>
write is success!

result for year 121 is <Response [200]>
write is success!

result for year 122 is <Response [200]>
write is success!

result for year 123 is <Response [200]>
write is success!

result for year 124 is <Response [200]>
write is success!

result for year 125 is <Response [200]>
write is success!

result for year 126 is <Response [200]>
write is success!



In [3]:
import json
import pandas as pd
import os

# Define the range of 'th' values (corresponding to years 2017-2026)
# The previous cell used initial_bound = 117 and end_bound = 127 (exclusive),
# meaning 'y' (th_value) ranged from 117 to 126.
start_th_year = 117
end_th_year = 126 # Corresponds to year 2026

all_collected_data = []

print("Starting data extraction for M1 and M2...")

for y_th in range(start_th_year, end_th_year + 1):
    # Calculate the actual year (e.g., 117 corresponds to 2017)
    year_actual = "20" + str(y_th)[1:3]

    # Construct the filename for the JSON file
    filename = f"SEK_Data_{year_actual}.json"
    file_path = f"/content/{filename}" # Assuming files are in /content/

    if not os.path.exists(file_path):
        print(f"Warning: JSON file not found for year {year_actual}. Skipping.")
        continue

    try:
        with open(file_path, 'r') as f:
            data = json.load(f)

        datacontent = data.get('datacontent', {})
        if not datacontent:
            print(f"Warning: 'datacontent' key not found or is empty in {filename}. Skipping year {year_actual}.")
            continue

        # Define the range of months (January to December)
        start_month = 1
        end_month = 12

        for month in range(start_month, end_month + 1):
            # The user requested 'M1 by value 1' and 'M2 by value 8'.
            # Based on the API's 'vervar' mapping, 'Jumlah (M1 = i+ii)' corresponds to 'val': 3
            # and 'Jumlah (M2 = M1 + v + vi)' corresponds to 'val': 8.
            # I will use '3' for M1 (as it represents the aggregate M1) and '8' for M2.
            # If 'Uang Kartal' (val 1) was specifically desired for M1, this can be adjusted.

            # Construct the keys for M1 and M2
            # Key format: [M1/M2 indicator][var_id=123][th_value_padded][month_value]
            th_value_padded = str(y_th).zfill(4)
            m1_key = f"3123{th_value_padded}{month}"
            m2_key = f"8123{th_value_padded}{month}"

            m1_value = datacontent.get(m1_key)
            m2_value = datacontent.get(m2_key)

            all_collected_data.append({
                'year': int(year_actual),
                'month': month,
                'M1': m1_value,
                'M2': m2_value
            })
        print(f"Successfully processed data for year {year_actual}")

    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {filename}. Skipping this file.")
    except Exception as e:
        print(f"An unexpected error occurred while processing {filename}: {e}. Skipping.")

if not all_collected_data:
    print("No data was extracted. Please check the JSON files and extraction logic.")
else:
    # Convert the list of dictionaries to a pandas DataFrame for a structured view
    df_m1m2 = pd.DataFrame(all_collected_data)

    print("\nSample of extracted data:")
    display(df_m1m2.head())

    print("\nMissing values in extracted data (if any):")
    display(df_m1m2.isnull().sum())

    # Save the collected data to a single JSON file
    output_filename = "M2M1_Data_2017_2026.json"
    with open(output_filename, 'w') as outfile:
        json.dump(all_collected_data, outfile, indent=4) # Use indent for pretty-printing

    print(f"\nAll collected M1 and M2 data saved to {output_filename}")

Starting data extraction for M1 and M2...
Successfully processed data for year 2017
Successfully processed data for year 2018
Successfully processed data for year 2019
Successfully processed data for year 2020
Successfully processed data for year 2021
Successfully processed data for year 2022
Successfully processed data for year 2023
Successfully processed data for year 2024
Successfully processed data for year 2025
Successfully processed data for year 2026

Sample of extracted data:


,year,month,M1,M2
0,2017,1,1191499.69,4936881.99
1,2017,2,1196036.61,4942919.76
2,2017,3,1215856.68,5017643.55
3,2017,4,1245927.39,5033780.29
4,2017,5,1275892.50,5125383.79



Missing values in extracted data (if any):


,0
year,0
month,0
M1,6
M2,6



All collected M1 and M2 data saved to M2M1_Data_2017_2026.json


# Change the file type from csv to json

In [ ]:
import pandas as pd
import json

# Define the input CSV file and output JSON file names
csv_input_file = "BI_Rate_2017_2026.csv"
json_output_file = "BI_Rate_2017_2026.json"

# Read the CSV file into a pandas DataFrame
try:
    df_bi_rate = pd.read_csv(csv_input_file)
    print(f"Successfully loaded {csv_input_file} into a DataFrame.")
    display(df_bi_rate.head())
except FileNotFoundError:
    print(f"Error: The file {csv_input_file} was not found. Please ensure it's in the correct directory.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

# Convert the DataFrame to JSON and save it to a file
# Using records orientation for a list of JSON objects (one per row)
# Using indent for pretty-printing
if 'df_bi_rate' in locals(): # Check if the DataFrame was successfully created
    try:
        df_bi_rate.to_json(json_output_file, orient="records", indent=4)
        print(f"\nSuccessfully converted and saved data to {json_output_file}")
    except Exception as e:
        print(f"An error occurred while writing the JSON file: {e}")

Successfully loaded BI_Rate_2017_2026.csv into a DataFrame.


,NO,Tanggal,BI Rate
0,1,19 Agustus 2026,5.75 %
1,2,22 Juli 2026,5.75 %
2,3,18 Juni 2026,5.75 %
3,4,9 Juni 2026,5.50 %
4,5,20 Mei 2026,5.25 %



Successfully converted and saved data to BI_Rate_2017_2026.json


# Extract Inflation M to M Data from BPS

In [7]:
import json

initial_bound = 117
end_bound = 127 #plus one for boundary

for y in range(initial_bound, end_bound):
  API_KEY = "..."
  year = y
  url = f"https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/0000/var/1/th/{year}/key/{API_KEY}"

  p = requests.get(url=url)
  print(f"result for year {year} is", p)

  if p.status_code == int(200):
    year_actual = "20" + str(year)[1:3]
    with open(f"Inflation_Data_{year_actual}.json", mode="w") as file:
      file.write(json.dumps(p.json(), indent=4))

  print("write is success!\n")

result for year 117 is <Response [200]>
write is success!

result for year 118 is <Response [200]>
write is success!

result for year 119 is <Response [200]>
write is success!

result for year 120 is <Response [200]>
write is success!

result for year 121 is <Response [200]>
write is success!

result for year 122 is <Response [200]>
write is success!

result for year 123 is <Response [200]>
write is success!

result for year 124 is <Response [200]>
write is success!

result for year 125 is <Response [200]>
write is success!

result for year 126 is <Response [200]>
write is success!



In [8]:
import json
import pandas as pd
import os

# Define the range of 'th' values (corresponding to years 2017-2026)
# The previous cell used initial_bound = 117 and end_bound = 127 (exclusive),
# meaning 'y' (th_value) ranged from 117 to 126.
start_th_year = 117
end_th_year = 126 # Corresponds to year 2026

all_collected_data = []

print("Starting data extraction for inflation...")

for y_th in range(start_th_year, end_th_year + 1):
    # Calculate the actual year (e.g., 117 corresponds to 2017)
    year_actual = "20" + str(y_th)[1:3]

    # Construct the filename for the JSON file
    filename = f"Inflation_Data_{year_actual}.json"
    file_path = f"/content/{filename}" # Assuming files are in /content/

    if not os.path.exists(file_path):
        print(f"Warning: JSON file not found for year {year_actual}. Skipping.")
        continue

    try:
        with open(file_path, 'r') as f:
            data = json.load(f)

        datacontent = data.get('datacontent', {})
        if not datacontent:
            print(f"Warning: 'datacontent' key not found or is empty in {filename}. Skipping year {year_actual}.")
            continue

        # Define the range of months (January to December)
        start_month = 1
        end_month = 12

        for month in range(start_month, end_month + 1):
            # The user requested 'M1 by value 1' and 'M2 by value 8'.
            # Based on the API's 'vervar' mapping, 'Jumlah (M1 = i+ii)' corresponds to 'val': 3
            # and 'Jumlah (M2 = M1 + v + vi)' corresponds to 'val': 8.
            # I will use '3' for M1 (as it represents the aggregate M1) and '8' for M2.
            # If 'Uang Kartal' (val 1) was specifically desired for M1, this can be adjusted.

            # Construct the keys for M1 and M2
            # Key format: [M1/M2 indicator][var_id=123][th_value_padded][month_value]
            th_value_padded = str(y_th).zfill(5)
            Indonesia_key = f"9999{th_value_padded}{month}"

            indonesia_val = datacontent.get(Indonesia_key)

            all_collected_data.append({
                'year': int(year_actual),
                'month': month,
                'value': indonesia_val,
            })
        print(f"Successfully processed data for year {year_actual}")

    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {filename}. Skipping this file.")
    except Exception as e:
        print(f"An unexpected error occurred while processing {filename}: {e}. Skipping.")

if not all_collected_data:
    print("No data was extracted. Please check the JSON files and extraction logic.")
else:
    # Convert the list of dictionaries to a pandas DataFrame for a structured view
    df_m1m2 = pd.DataFrame(all_collected_data)

    print("\nSample of extracted data:")
    display(df_m1m2.head())

    print("\nMissing values in extracted data (if any):")
    display(df_m1m2.isnull().sum())

    # Save the collected data to a single JSON file
    output_filename = "Inflation_MtoM_2017_2026.json"
    with open(output_filename, 'w') as outfile:
        json.dump(all_collected_data, outfile, indent=4) # Use indent for pretty-printing

    print(f"\nAll collected M1 and M2 data saved to {output_filename}")

Starting data extraction for inflation...
Successfully processed data for year 2017
Successfully processed data for year 2018
Successfully processed data for year 2019
Successfully processed data for year 2020
Successfully processed data for year 2021
Successfully processed data for year 2022
Successfully processed data for year 2023
Successfully processed data for year 2024
Successfully processed data for year 2025
Successfully processed data for year 2026

Sample of extracted data:


,year,month,value
0,2017,1,None
1,2017,2,None
2,2017,3,None
3,2017,4,None
4,2017,5,None



Missing values in extracted data (if any):


,0
year,0
month,0
value,120



All collected M1 and M2 data saved to Inflation_MtoM_2017_2026.json


# Delete specific file


In [9]:
#delete file!

import os

# Reuse the same bounds as before
# initial_bound = 117
# end_bound = 2027 # This was set for fetching, check if this range is desired for deletion

# Adjust the range if you only want to delete files that were actually created in the previous step
# For example, if the previous execution was interrupted, the files might not exist for the full range.
# Assuming you want to delete files for the range 117 to 145 (inclusive) based on the last run's output.

# For demonstration, I will use a smaller range that was definitely processed.
# If you want to delete all files from initial_bound to end_bound-1, uncomment the original bounds.

delete_initial_bound = initial_bound
delete_end_bound = 127 # Based on the last successful write in the output

print(f"Attempting to delete files from year index {delete_initial_bound} to {delete_end_bound-1}")

for y in range(delete_initial_bound, delete_end_bound):
  year_actual = "20" + str(y)[1:3] # Recalculate year_actual based on the index
  filename = f"Inflation_Data_{year_actual}.json"
  if os.path.exists(filename):
    os.remove(filename)
    print(f"Deleted file: {filename}")
  else:
    print(f"File not found: {filename}")

print("File deletion process complete.")

Attempting to delete files from year index 117 to 126
Deleted file: Inflation_Data_2017.json
Deleted file: Inflation_Data_2018.json
Deleted file: Inflation_Data_2019.json
Deleted file: Inflation_Data_2020.json
Deleted file: Inflation_Data_2021.json
Deleted file: Inflation_Data_2022.json
Deleted file: Inflation_Data_2023.json
Deleted file: Inflation_Data_2024.json
Deleted file: Inflation_Data_2025.json
Deleted file: Inflation_Data_2026.json
File deletion process complete.


In [12]:
import pandas as pd
import json

# Define the input CSV file and output JSON file names
csv_input_file = "Inflation_YoY_2017_2026.csv"
json_output_file = "Inflation_YoY_2017_2026.json"

# Read the CSV file into a pandas DataFrame
try:
    df_inflation_yoy_csv = pd.read_csv(csv_input_file)
    print(f"Successfully loaded {csv_input_file} into a DataFrame.")
    display(df_inflation_yoy_csv.head())
except FileNotFoundError:
    print(f"Error: The file {csv_input_file} was not found. Please ensure it's in the correct directory.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

# Convert the DataFrame to JSON and save it to a file
# Using records orientation for a list of JSON objects (one per row)
# Using indent for pretty-printing
if 'df_inflation_yoy_csv' in locals(): # Check if the DataFrame was successfully created
    try:
        df_inflation_yoy_csv.to_json(json_output_file, orient="records", indent=4)
        print(f"\nSuccessfully converted and saved data to {json_output_file}")
    except Exception as e:
        print(f"An error occurred while writing the JSON file: {e}")

Successfully loaded Inflation_YoY_2017_2026.csv into a DataFrame.


,No,Periode,Inflasi
0,1,Juli 2026,2.88 %
1,2,Juni 2026,3.34 %
2,3,Mei 2026,3.08 %
3,4,April 2026,2.42 %
4,5,Maret 2026,3.48 %



Successfully converted and saved data to Inflation_YoY_2017_2026.json
